In [ ]:
import numpy as np
import pandas as pd

In [ ]:
# Data
T1 = 160 #hot fluid inlet temperature
T2 = 120 #hot fluid outlet temperature
t1 = 75 #cold fluid inlet temperature
t2 = 120 #cold fluid outlet temperature
R_dg = 5E-04 #Fouling factor - Gasoline (Hot fluid)
R_dk = 1E-03 #Fouling factor - Kerosene (Cold fluid)
P_inlet_g = 50 #Inlet pressure - Gasoline (Hot fluid)
P_inlet_k = 50 #Inlet pressure - Kerosene (Cold fluid)
delP_max_g = 7 #Max pressure drop - Gasoline (Hot fluid)
delP_max_k = 10 #Max pressure drop - Kerosene (Cold fluid)
m_k = 15E+04 #Mass flow rate of cold fluid

mu_g = 0.484 #Viscosity of gasoline
mu_k = 3.872 #Viscosity of kerosene

rho_g = 42.7 #Density of gasoline
rho_k = 49.8 #Density of kerosene

k_g = 0.075 #Thermal conductivity of gasoline
k_k = 0.083 #Thermal conductivity of kerosene

C_g = 0.57 #Specific heat capacity of gasoline
C_k = 0.48 #Specific heat capacity of kerosene

S_g = 0.685 #Specific gravity of gasoline
S_k = 0.8 #Specific gravity of kerosene

#Caloric temperature calc parameters
Kc_g = 1
Fc_g = 0.455

#Dimensions - tube side
d_o = 0.083 #outer diameter in ft
p = 0.104167 #pitch in ft
flow_area_per_tube = 0.546 / 144.0
L_t = 24 #length of tube in inches
d_i = 0.07 #inner diameter in ft
k_w = 70 #Thermal conductivity of brass (tube material)

U_assm = 45 #Assumed overall heat transfer coefficient (Btu h-1 ft2 deg F )
n_p = 6 #tube passes
n_s = 1 #shell passes

#Dimensions - shell side
Ds = 2.58 #shell side diameter in ft
B = 0.5*Ds

In [ ]:
# Calorific temperatures
r = (T2 - t1)/(T1 - t2)
Thc = T2 + Fc_g *(T1 - T2)
Tcc = t1 + Fc_g *(T2 - t1)
print("Caloric temperature of hot fluid : ", Thc)
print("Caloric temperature of cold fluid : ",Tcc)

Caloric temperature of hot fluid :  138.2
Caloric temperature of cold fluid :  95.475


In [ ]:
#Energy Balance:
def energybalance(m_k, C_k, t1, t2, C_g, T1, T2):
    Q_k = m_k * C_k * (t2-t1)
    m_g = Q_k / (C_g * (T1-T2))
    return Q_k, m_g

Q_k, m_g = energybalance(m_k, C_k, t1, t2, C_g, T1, T2)
print("Heat load: ",Q_k,"Btu/h")
print("Mass flowrate of Gasoline : ",m_g,"lb/h")

Heat load:  3240000.0 Btu/h
Mass flowrate of Gasoline :  142105.26315789475 lb/h


In [ ]:
# Correction factors
LMTD = ((T1 - T2) - (t2 - t1)) / np.log((T1 - T2) / (t2 - t1))
print(f"LMTD = {LMTD:.4f}")
R = (T1 - T2) / (t2 - t1)
S = (t2 - t1) / (T1 - t1)
print("R = ", R)
print("S =" , S)

# Correction factor
Ft = (((R**2 + 1)**0.5 * np.log((1 - S) / (1 - R * S)))) / ((R - 1) * np.log((2 - S * (R + 1 - (R**2 + 1)**0.5)) / (2 - S * (R + 1 + (R**2 + 1)**0.5))))

print("Ft = ", Ft)

LMTD = 42.4509
R =  0.8888888888888888
S = 0.5294117647058824
Ft =  0.8023645281937324


In [ ]:
A = Q_k / (U_assm * LMTD * Ft)
print(f"Heat transfer area (A): {A:.2f} ft2")

def nt(L_t_ft):
    if L_t_ft == 0:
        return 0
    return A / (np.pi * d_o * L_t_ft)

def Re(n_p_val, L_t_ft):
    count = nt(L_t_ft)
    if count == 0:
        return 0
    return 4 * m_k * (n_p_val / count) / (np.pi * d_i * mu_k)

L = [16,24]
n_p = 2
Re_t = 0
L_t_opt = 0
found = False

while n_p < 20 and not found:
    for i in L:
        current_L_t = i
        current_Re = Re(n_p, current_L_t)

        # Target fully turbulent flow (>10,000)
        if current_Re > 10000:
            Re_t = current_Re
            L_t_opt = current_L_t
            found = True
            break

    if not found:
        n_p += 2  # Increment by 2 to keep tube passes even (2, 4, 6...


print(f"\n Optimization Layout Found ")
print(f"Optimal Tube Passes (n_p)      : {n_p}")
print(f"Optimal Tube Length (L_t)      : {L_t_opt} ft")
print(f"Reynolds Number (Re)           : {Re_t:.2f}")

# Calculate velocities (Since d_i is already in ft, remove the /12)
u_t = (Re_t * mu_k / (d_i * rho_k)) / 11811
print(f"Tube side velocity             : {u_t:.4f} m/s")

# Actual integer count of tubes
nt_actual = int(np.ceil(nt(L_t_opt)))
print(f"Actual Number of tubes needed  : {nt_actual}")

Heat transfer area (A): 2113.85 ft2

 Optimization Layout Found 
Optimal Tube Passes (n_p)      : 6
Optimal Tube Length (L_t)      : 24 ft
Reynolds Number (Re)           : 12516.56
Tube side velocity             : 1.1771 m/s
Actual Number of tubes needed  : 338


In [ ]:
# Heat transfer coefficients on tube and shell side
# Assumptions- square pitch of tubes, 25% segmented baffles
Re_t= 11571.4 # tube-side fluid Reynolds number
jH_t= 42 # Colburn j-factor for tube-side fluid (kerosene)
phi = 1 # ratio of tube-side viscosity to viscosity of fluid at wall temperature

def tubeside_htc(jH, Re, phi) :
    htc_t= (jH_t*k_k/d_i)*((mu_k*C_k/k_k)**(1/3))
    return htc_t

h_i = tubeside_htc(jH_t, Re_t, phi) # heat transfer coefficient on tube-side
print(f"Tube-side heat transfer coefficient:{h_i:.3f} Btu/(h-ft-F)")

# Shell-side heat transfer coefficient
Pt= 1.25/12 # ft
do= 1/12 # ft
C = (Pt-do)# tube clearance (ft)

def equivalent_dia(Pt, C, do):
    De= (4*((Pt**2)-(np.pi/4)*(do**2)))/(np.pi*do)
    return De

De= equivalent_dia(Pt,C,do)
print(f"Equivalent diameter:{De:.3f} ft")

def shellside_area(C):
    a_s= C*B*Ds/Pt
    return a_s
a_s= shellside_area(C) # shellside area
print(f"shellside area:{a_s:.6f} sqft")

jH_s= 110 # Colburn j-factor
Gs= m_g/a_s # mass velocity of gasoline
Re_s= De*Gs/mu_g # Reynolds number of gasoline

def shellside_htc(jH, Re, phi) :
    h_o = (jH_s*k_g/De)*((mu_g*C_g/k_g)**(1/3))
    return h_o

h_o = shellside_htc(jH_s, Re_s, phi) # heat transfer coefficient on shell-side
print(f"Shell-side heat transfer coefficient:{h_o:.3f} Btu/(h-ft-F)")

Tube-side heat transfer coefficient:140.366 Btu/(h-ft-F)
Equivalent diameter:0.082 ft
shellside area:0.665640 sqft
Shell-side heat transfer coefficient:154.454 Btu/(h-ft-F)


In [ ]:
# CALCULATION OF OVERALL HEAT TRANSFER COEFFICIENT (Uo)
Ao_Ai = d_o / d_i # area ratio
outside_convection_resistance = 1 / h_o
gasoline_fouling = R_dg
wall_conductive_resistance = ((d_o - d_i) / (2 * k_w)) * Ao_Ai
kerosene_fouling = R_dk * Ao_Ai
inside_convection_resistance = (1 / h_i) * Ao_Ai

total_resistance = (
    outside_convection_resistance
    + gasoline_fouling
    + wall_conductive_resistance
    + kerosene_fouling
    + inside_convection_resistance
)

Uo_cal = 1 / total_resistance

print("Overall HTC, Uo =", Uo_cal, "Btu/(h ft^2 °F)")

Overall HTC, Uo = 59.81748226939096 Btu/(h ft^2 °F)


In [ ]:
# Pressure drop calculations
N_t = 368  # Actual selected tube count
# Tube-Side Pressure Drop
f_t = 0.00028 * 144.0
a_t = (N_t * flow_area_per_tube) / n_p
G_t = m_k / a_t

delP_t = (f_t * (G_t**2) * L_t * n_p) / (7.5e12 * d_i * S_k * 1.0)
delP_rt = 1.334e-13 * (2 * n_p - 1.5) * (G_t**2) / S_k
delP_T = delP_t + delP_rt

# Shell-Side Pressure Drop
G_s = m_g / a_s
n_b = round(L_t / B)
f_s = 0.0017 * 144.0
# Shell-side pressure drop
G_s = m_g / a_s
n_b = round(L_t / B)
f_s = 0.0017 * 144.0

delP_s = (f_s * (G_s**2) * Ds * (n_b + 1)) / (7.5e12 * De * S_g * 1.0)
delP_rs = 0.0
delP_S = delP_s + delP_rs

print(f"Tube Flow Area (a_t)         : {a_t:.4f} sq ft")
print(f"Frictional Tube Drop (delP_t): {delP_t:.2f} psi")
print(f"Return Loss (delP_rt)        : {delP_rt:.2f} psi")
print(f"Total Tube Side Drop (delP_T): {delP_T:.2f} psi")
print(f"Total Shell Side Drop (delP_S): {delP_S:.2f} psi")

Tube Flow Area (a_t)         : 0.2326 sq ft
Frictional Tube Drop (delP_t): 5.75 psi
Return Loss (delP_rt)        : 0.73 psi
Total Tube Side Drop (delP_T): 6.48 psi
Total Shell Side Drop (delP_S): 1.36 psi


In [ ]:
# Over surface
h_io = h_i * (d_i / d_o)
Uc = h_o * h_io / (h_o + h_io)
over_surface = (Uc - Uo_cal) / Uc
pct_over_surface = ((Uc - Uo_cal) / Uc) * 100
print(f"Percentage Oversurface = {pct_over_surface:.2f}%")

# Over Design based on actual provided tube count vs required
n_t_provided = nt_actual  # 338 tubes
n_t_required = A / (np.pi * d_o * L_t)

A_provided = np.pi * d_o * L_t * n_t_provided
A_req = np.pi * d_o * L_t * n_t_required

A_provided = np.pi * d_o * L_t * N_t
Over_design = ((A_provided - A_req) / A_req) * 100.0
print(f"Percentage Overdesign: {Over_design:.2f}%")

Percentage Oversurface = 10.74%
Percentage Overdesign: 8.95%


In [ ]:
# Create a dictionary of all key calculated results
results_data = {
    "Parameter": [
        "Caloric Temp (Hot Fluid)",
        "Caloric Temp (Cold Fluid)",
        "Heat Load (Q)",
        "Mass Flowrate Gasoline",
        "LMTD",
        "Correction Factor (Ft)",
        "Heat Transfer Area (A)",
        "Tube Length (L_t)",
        "Tube Passes (n_p)",
        "Number of Tubes (N_t)",
        "Tube Reynolds Number (Re_t)",
        "Tube-Side HTC (h_i)",
        "Equivalent Diameter (De)",
        "Shell-Side Area (a_s)",
        "Shell-Side HTC (h_o)",
        "Overall HTC Calculated (Uo_cal)",
        "Tube Flow Area (a_t)",
        "Tube Mass Velocity (G_t)",
        "Tube Frictional Drop (delP_t)",
        "Tube Return Loss (delP_rt)",
        "Total Tube Pressure Drop (delP_T)",
        "Shell Mass Velocity (G_s)",
        "Shell Reynolds Number (Re_s)",
        "Total Shell Pressure Drop (delP_S)",
        "Percent Over Surface",
        "Percent Overdesign"
    ],
    "Value": [
        Thc,
        Tcc,
        Q_k,
        m_g,
        LMTD,
        Ft,
        A,
        L_t,
        n_p,
        nt(L_t),
        Re_t,
        h_i,
        De,
        a_s,
        h_o,
        Uo_cal,
        a_t,
        G_t,
        delP_t,
        delP_rt,
        delP_T,
        G_s,
        Re_s,
        delP_S,
        pct_over_surface,
        Over_design
    ],
    "Unit": [
        "°F",
        "°F",
        "Btu/h",
        "lb/h",
        "°F",
        "-",
        "sq ft",
        "ft",
        "-",
        "-",
        "-",
        "Btu/(h·ft²·°F)",
        "ft",
        "sq ft",
        "Btu/(h·ft²·°F)",
        "Btu/(h·ft²·°F)",
        "sq ft",
        "lb/(h·ft²)",
        "psi",
        "psi",
        "psi",
        "lb/(h·ft²)",
        "-",
        "psi",
        "%",
        "%"
    ]
}

# Create DataFrame
df_results = pd.DataFrame(results_data)

# Format numerical values to 4 decimal places for clean display
df_results["Value"] = df_results["Value"].apply(lambda x: f"{x:.4f}")

# Display the summary table
print("\n" + "="*55)
print("       HEAT EXCHANGER DESIGN CALCULATION SUMMARY       ")
print("="*55)
print(df_results.to_string(index=False))


       HEAT EXCHANGER DESIGN CALCULATION SUMMARY       
                         Parameter        Value           Unit
          Caloric Temp (Hot Fluid)     138.2000             °F
         Caloric Temp (Cold Fluid)      95.4750             °F
                     Heat Load (Q) 3240000.0000          Btu/h
            Mass Flowrate Gasoline  142105.2632           lb/h
                              LMTD      42.4509             °F
            Correction Factor (Ft)       0.8024              -
            Heat Transfer Area (A)    2113.8468          sq ft
                 Tube Length (L_t)      24.0000             ft
                 Tube Passes (n_p)       6.0000              -
             Number of Tubes (N_t)     337.7803              -
       Tube Reynolds Number (Re_t)   11571.4000              -
               Tube-Side HTC (h_i)     140.3661 Btu/(h·ft²·°F)
          Equivalent Diameter (De)       0.0825             ft
             Shell-Side Area (a_s)       0.6656          sq f